In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1" # disable XEt for JupyterHub
from dotenv import load_dotenv
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel, load_from_disk, concatenate_datasets
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
#import evaluate
from scipy.special import softmax
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import random
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import gc

SEED = 40

random.seed(SEED)
load_dotenv()

True

In [2]:
def resplit(DS_PATH, SAVE_PATH):
    ds = load_from_disk(DS_PATH)

    if isinstance(ds, DatasetDict):
        target_features = ClassLabel(names=["0", "1"])
        
        normalized_splits = []
        for split_name in ds.keys():
            split_ds = ds[split_name]
            if split_ds.features["label"] != target_features:
                split_ds = split_ds.cast_column("label", target_features)
            normalized_splits.append(split_ds)
        
        #ds = concatenate_datasets([ds[split] for split in ds.keys()])
        ds = concatenate_datasets(normalized_splits)

    train_testvalid = ds.train_test_split(
        test_size=0.2, 
        seed=SEED, 
        stratify_by_column="label"
    )
    
    test_valid = train_testvalid["test"].train_test_split(
        test_size=0.5, 
        seed=SEED, 
        stratify_by_column="label"
    )

    final_ds = DatasetDict({
        "train": train_testvalid["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"]
    })

    # --- Save onegaishimasu ---

    final_ds.save_to_disk(SAVE_PATH)

    return final_ds

In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if np.isnan(logits).any() or np.isinf(logits).any():
        logits = np.nan_to_num(
            logits,
            nan=0.0,
            posinf=100.0,
            neginf=-100.0
        )

    preds = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=-1)[:, 1] # only harmful label

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    fpr = fp / (tn + fp) if (tn + fp) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    precision, recall, f1, support = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0.0)

    acc = accuracy_score(labels, preds)
    try:
        roc_auc = roc_auc_score(labels, probs)
    except ValueError:
        roc_auc = 0.5

    metrics = {
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": roc_auc,
    "fpr": fpr,
    "tnr": tnr,
    }

    return metrics

In [4]:
def shutil_helper(data_name):
    import shutil
    shutil.make_archive(data_name, "zip", data_name)

In [5]:
def process_dataset(
    dataset_name: str,
    text_col: str,
    label_col: str,
    default_label: int = None,
    split: str = "train",
) -> pd.DataFrame:

    ds = load_dataset(dataset_name)
    df = pd.DataFrame(ds[split])

    texts = df[text_col]

    # Process labels ...
    if default_label is not None:
        labels = default_label
    else: ...# labelmap or just use column?        

    return pd.DataFrame({
    "text": texts,
    "label": labels,
    "soruce": dataset_name
    })

In [6]:
# candidate https://huggingface.co/datasets/bogdanminko/Catch_the_prompt_injection_or_jailbreak_or_benign/viewer/default/train?views%5B%5D=train

def combine_jb_datasets():
    processed_dss = []

    # --- JACKHAO --- 
    ds_jack = load_dataset("jackhhao/jailbreak-classification")
    df_jack = pd.DataFrame(ds_jack["train"]) 
    
    label_map = {"jailbreak": 1, "benign": 0}
    df_jack["label"] = df_jack["type"].map(label_map)
    df_jack = df_jack[["prompt", "label"]].rename(columns={"prompt": "text"})
    df_jack["source"] = "jackhao/jailbreak-classification"
    processed_dss.append(df_jack)

    # --- SEVDEVAWESOME ----

    ds_sev = load_dataset("sevdeawesome/jailbreak_success")
    df_sevdev = pd.DataFrame(ds_sev["train"])

    df_jailbreak = pd.DataFrame({
        "text": df_sevdev["jailbreak_prompt_text"],
        "label": 1,
        "source": "sevdeawesome/jailbreak_success"
    })


    processed_dss.append(df_jailbreak)

    # --- TrustAIRLab ----

    ds_trust_jb = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", "jailbreak_2023_12_25")
    df_trust_jb = pd.DataFrame(ds_trust_jb["train"])
    df_trust_jb_final = pd.DataFrame({
        "text": df_trust_jb["prompt"],
        "label": 1,
        "source": "TrustAIRLab/in-the-wild-jailbreak-prompts"
    })

    ds_trust_regular = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", "regular_2023_12_25")
    df_trust_regular = pd.DataFrame(ds_trust_regular["train"])
    df_trust_regular = df_trust_regular.sample(n=4500, random_state=SEED)
    # Add some benign examples 
    df_trust_reg_final = pd.DataFrame({
        "text": df_trust_regular["prompt"],
        "label": 0,
        "source": "TrustAIRLab/in-the-wild-jailbreak-prompts"
    })

    df_trust_final = pd.concat([df_trust_jb_final, df_trust_reg_final], ignore_index=True)
    processed_dss.append(df_trust_final)

    # ---- Deepset ----

    ds_deepset = load_dataset("deepset/prompt-injections")
    df_deepset = pd.DataFrame(ds_deepset["train"])
    df_deepset["source"] = "deepset/prompt-injection"
    processed_dss.append(df_deepset)

    # --- Add Hard Negatives through SQuAD, Better Alternatives? ---
    """
    ds_squad = load_dataset("rajpurkar/squad")
    df_squad = pd.DataFrame(ds_squad["train"][:5000])
    df_squad_final = pd.DataFrame({
        "text": df_squad["question"],
        "label": 0,
        "source": "rajpukar/squad"
    })
    processed_dss.append(df_squad_final)
    """

    ds_poly = load_dataset("ToxicityPrompts/PolyGuardPrompts")
    df_poly = pd.DataFrame(ds_poly["test"])
    df_poly = df_poly[df_poly["language"] == "English"]
    df_poly = df_poly[df_poly["prompt_label"] == "safe"]
    df_poly_final = pd.DataFrame({
        "text": df_poly["prompt"],
        "label": 0,
        "source": "ToxicityPrompts/PolgyGuardPrompts"
    }) # 1000 benign
    
    # --- Combine ---

    full_df = pd.concat(processed_dss, ignore_index=True)
    full_df = full_df.drop_duplicates(subset=["text"]) # !!!

    combined_hf = Dataset.from_pandas(full_df, preserve_index=False)
    combined_hf = combined_hf.cast_column(
        "label",
        ClassLabel(names=["0", "1"])
    )

    train_testvalid = combined_hf.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
    test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="label")

    final_ds = DatasetDict({
        "train": train_testvalid["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"]
    })

    # --- Save onegaishimasu ---

    output_path = "./combined_input"
    final_ds.save_to_disk(output_path)

    return final_ds

def ds_sanitycheck(dataset):
    all_labels = dataset["train"]["label"]
    flat = np.array(all_labels)
    print(f"Min {flat.min()}")
    print(f"Max {flat.max()}")
    print(f"Shape {flat.shape}")
    print(f"Label 1 {np.sum(flat == 1)}") # 67%
    print(f"Label 0 {np.sum(flat == 0)}") # 33%

    for k in dataset["train"].column_names:
        print(k)

    print(Counter(dataset["train"]["label"]))
    print(Counter(dataset["validation"]["label"]))
    print(Counter(dataset["test"]["label"]))

    labels = defaultdict(set)

    for text, label in zip(dataset["train"]["text"], dataset["train"]["label"]):
        labels[text].add(label)
    
    conflicts = {text: label for text, label in labels.items() if len(label) > 1}
    
    print(f"Conflicts: {len(conflicts)}")


In [10]:
def combine_output_datasets():
    # 1 is UNSAFE and 0 is SAFE
    # https://huggingface.co/datasets/nicholasKluge/harmless-aira-dataset
    # Split: English, rejected responses -> 

    # https://huggingface.co/datasets/PKU-Alignment/BeaverTails
    # is_safe false vs true -> 330k train, 330k test

    # https://huggingface.co/datasets/PKU-Alignment/self-monitor
    # two responses, both can be false or true ? -> only train

    # hard negatives 

    processed_dss = []

    # === BEAVERTAILS ===
    # Maybe switch to smaller version ?
    str_beavertails = "PKU-Alignment/BeaverTails"
    ds_beavertails = load_dataset(str_beavertails)
    df_beavertails = pd.DataFrame(ds_beavertails["330k_train"]) 
    
    df_beavertails["is_safe"] = df_beavertails["is_safe"].map({True: 0, False: 1})
    print(df_beavertails["is_safe"][:10])

    df_beavertails_final = pd.DataFrame({
    "text": df_beavertails["response"],
    "label": df_beavertails["is_safe"],
    "source": str_beavertails
    })
    processed_dss.append(df_beavertails_final)

    # === HARMLESS AIRA ===

    str_harmless = "nicholasKluge/harmless-aira-dataset"
    ds_harmless = load_dataset(str_harmless)
    df_harmless = pd.DataFrame(ds_harmless["english"])
    #print(df_harmless.columns)
    df_harmless_final = pd.DataFrame({
        "text": df_harmless["rejected_response"],
        "label": 1,
        "source": str_harmless
    })

    processed_dss.append(df_harmless_final)

    # === self-monitor ===
    str_monitor = "PKU-Alignment/self-monitor"
    ds_monitor = load_dataset(str_monitor)
    df_monitor = pd.DataFrame(ds_monitor["train"])
    df_monitor["is_response_0_safe"] = df_monitor["is_response_0_safe"].map({True: 0, False: 1})
    df_monitor["is_response_1_safe"] = df_monitor["is_response_1_safe"].map({True: 0, False: 1})
    print(df_monitor["is_response_0_safe"][:10])
    df_monitor_final_0 = pd.DataFrame({
        "text": df_monitor["response_1"],
        "label": df_monitor["is_response_0_safe"],
        "source": str_monitor
    })
    df_monitor_final_1 = pd.DataFrame({
        "text": df_monitor["response_2"],
        "label": df_monitor["is_response_1_safe"],
        "source": str_monitor
    })

    processed_dss.append(df_monitor_final_0)
    processed_dss.append(df_monitor_final_1)

    # --- Combine ---

    full_df = pd.concat(processed_dss, ignore_index=True)
    full_df = full_df.drop_duplicates(subset=["text"]) # !!!

    combined_hf = Dataset.from_pandas(full_df, preserve_index=False)
    combined_hf = combined_hf.cast_column(
        "label",
        ClassLabel(names=["0", "1"])
    )

    train_testvalid = combined_hf.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
    test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="label")

    final_ds = DatasetDict({
        "train": train_testvalid["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"]
    })

    # --- Save onegaishimasu ---

    output_path = "./combined_harm_datasets"
    final_ds.save_to_disk(output_path)
    

In [11]:
def augment_ds_2(dataset, SAVE_PATH):
    #dataset = load_from_disk(DS_PATH)
    """
    char_aug = nac.KeyboardAug(aug_char_p=0.1, aug_word_p=0.2)
    leet_aug = nac.RandomCharAug(action="substitute", aug_char_p=0.15) 
    del_aug = nac.RandomCharAug(action="delete", aug_char_p=0.15)
    ins_aug = nac.RandomCharAug(action="insert", aug_char_p=0.15)
    word_aug = naw.SynonymAug(aug_src="wordnet", aug_p=0.15) # old 0.15
    harmful_augs = [char_aug, leet_aug, del_aug, ins_aug]
    """
    #leet_aug = nac.RandomCharAug(action="substitute", aug_char_p=0.15)
    # context_aug but uses BERT?
    # antonym_aug = naw.AntonymAug(aug_p=0.2)

    context_aug = naw.ContextualWordEmbsAug(
        model_path='distilroberta-base',
        action='substitute',
        aug_p=0.15,
        device='cuda',
        batch_size=8,
        use_custom_api=False
    )

    back_translate_aug = naw.BackTranslationAug(
        from_model_name='facebook/wmt19-en-de',
        to_model_name='facebook/wmt19-de-en',
        device='cuda',
        batch_size=8
    )

    augments = [context_aug, back_translate_aug]
    def augment(batch):
        augmented_texts = list(batch["text"])
        augmented_labels = list(batch["label"])
        augmented_source = list(batch["source"])

        benign_id = [i for i, label in enumerate(batch["label"]) if label == 0]

        if benign_id:
            benign_texts = [batch["text"][i] for i in benign_id]
            benign_sources = [batch["source"][i] for i in benign_id]

            # run context
            results = context_aug.augment(benign_texts)
            for old_text, new_text, source in zip(benign_texts, results, benign_sources):
                if new_text != old_text:
                    augmented_texts.append(new_text)
                    augmented_labels.append(0)
                    augmented_source.append(f"{source}_nlpaug")

            # run backtranslate
            results = back_translate_aug.augment(benign_texts)
            for old_text, new_text, source in zip(benign_texts, results, benign_sources):
                if new_text != old_text:
                    augmented_texts.append(new_text)
                    augmented_labels.append(0)
                    augmented_source.append(f"{source}_nlpaug")

            torch.cuda.empty_cache()

        return {
            "text": augmented_texts,
            "label": augmented_labels,
            "source": augmented_source
            }

    augmented_train = dataset["train"].map(
        augment,
        batched=True,
        batch_size=8,
        remove_columns=dataset["train"].column_names # delete old table structure
        )

    df_aug = augmented_train.to_pandas()
    # obsolte ?
    df_aug = df_aug.drop_duplicates(subset=["text"])

    dataset["train"] = Dataset.from_pandas(df_aug, preserve_index=False)

    dataset.save_to_disk(SAVE_PATH)
    print(len(dataset["train"]))
    return dataset

In [12]:
def augment_ds(DS_PATH, SAVE_PATH):
    probability = 0.24
    dataset = load_from_disk(DS_PATH)

    char_aug = nac.KeyboardAug(aug_char_p=0.1, aug_word_p=0.2)
    leet_aug = nac.RandomCharAug(action="substitute", aug_char_p=0.15) 
    del_aug = nac.RandomCharAug(action="delete", aug_char_p=0.15)
    ins_aug = nac.RandomCharAug(action="insert", aug_char_p=0.15)
    word_aug = naw.SynonymAug(aug_src="wordnet", aug_p=0.15) # old 0.15
    harmful_augs = [char_aug, leet_aug, del_aug, ins_aug]
    #leet_aug = nac.RandomCharAug(action="substitute", aug_char_p=0.15)
    # context_aug but uses BERT?
    # antonym_aug = naw.AntonymAug(aug_p=0.2)

    def augment(batch):
        augmented_texts = []
        augmented_labels = []
        augmented_source = []

        for text, label, source in zip(batch["text"], batch["label"], batch["source"]):
            augmented_texts.append(text)
            augmented_labels.append(label)
            augmented_source.append(source)
            
            if label == 1:
                if random.random() < 0.probability:
                    aug = random.choice(harmful_augs)
                    char_text = aug.augment(text)
                    if char_text != text:
                        augmented_texts.append(char_text[0])
                        augmented_labels.append(label)
                        augmented_source.append(f"{source}_nlpaug") 

            elif label == 0:
                if random.random() < probability:
                    word_text = word_aug.augment(text)
                    if word_text != text:
                        augmented_texts.append(word_text[0])
                        augmented_labels.append(label)
                        augmented_source.append(f"{source}_nlpaug")
            
            elif label == 0:
                if random.random() < probabilty:
                    word_text = word_aug.augment(text)
                    if word_text != text:
                        augmented_texts.append(word_text[0])
                        augmented_labels.append(label)
                        augmented_source.append(f"{source}_nlpaug")
        return {
            "text": augmented_texts,
            "label": augmented_labels,
            "source": augmented_source
        }

    augmented_train = dataset["train"].map(
        augment,
        batched=True,
        batch_size=1000,
        remove_columns=dataset["train"].column_names # delete old table structure
    )

    df_aug = augmented_train.to_pandas()
    # obsolte ?
    df_aug = df_aug.drop_duplicates(subset=["text"])

    dataset["train"] = Dataset.from_pandas(df_aug, preserve_index=False)

    dataset.save_to_disk(SAVE_PATH)
    print(len(dataset["train"]))
    return dataset
    

In [13]:
def train_model(MODEL_NAME="FacebookAI/roberta-base", DATA_SET="./combined_harm_aug", OUTPUT_PATH="./roberta_output_aug_smooth", EPOCHS=2):
    #MODEL_NAME = "microsoft/deberta-v3-base"
    #MODEL_NAME = "FacebookAI/roberta-base"    
    
    # === LOAD BERT ===
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=MODEL_NAME,
        num_labels=2, # Single output logit jb vs benign
        problem_type="single_label_classification",
        use_safetensors=True, # DEBERTA
    )
        
    # === TOKENIZATION ===
    def tokenize_helper(splits):
        return tokenizer(
            splits["text"],
            truncation=True,
            max_length=512
        )

    dataset = load_from_disk(DATA_SET)
    
    tokenized_dataset = dataset.map(tokenize_helper, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # === TRAINING ===
    training_args = TrainingArguments(
        output_dir=OUTPUT_PATH,
        learning_rate=2e-6,
        #learning_rate=2e-5, # too high for deberta
        #learning_rate=1e-5, # deberta
        #weight_decay=0.1, # too aggresive?
        warmup_ratio = 0.1, # switch to steps ?
        max_grad_norm=1.0, # default
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        label_smoothing_factor=0.1, # starting at 0.1
        num_train_epochs=EPOCHS,
        eval_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=250,
        save_total_limit=2,
        metric_for_best_model="f1",
        dataloader_pin_memory=False,
        #train_sampling_strategy="group_by_length",
        #fp16=True,
        bf16=True, # deberta suppors this, but not nvidia t4
        seed=SEED,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
        compute_metrics = compute_metrics
    )
        
    trainer_results = trainer.train()
    trainer.save_model(OUTPUT_PATH)
    trainer.save_metrics("train", trainer_results.metrics)

In [17]:
def evaluate_model(MODEL_NAME, OUTPUT_DIR, DATA_SET="./combined_harm_aug"):
    #MODEL_NAME = "roberta-jailbreak/checkpoint-6090"
    #MODEL_NAME = "protectai/deberta-v3-base-prompt-injection-v2"
    #MODEL_NAME = "shashidharbabu/roberta-jailbreak-guardrails"
    #MODEL_NAME = "pmking27/jailbreak-detection"
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
     # Qwen is missing pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    if "pmking27" in MODEL_NAME: # Double check if needed anymore?
        model = AutoModelForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path=MODEL_NAME,
            num_labels=2, # Single output logit jb vs benign
            problem_type="single_label_classification",
            #use_safetensors=True, # needed for sota model on JuptyerHub
            force_download=True,
        )
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=MODEL_NAME,
        num_labels=2, # Single output logit jb vs benign
        problem_type="single_label_classification",
        use_safetensors=True, # needed for sota model on JuptyerHub
        force_download=True,
        )


    # Explict set the token for Qwen
    model.config.pad_token_id = tokenizer.pad_token_id
    
    def tokenize_test(text):
        return tokenizer(
            text["text"], 
            truncation=True, 
            max_length=512, 
        )

    dataset = load_from_disk(DATA_SET)
    test_dataset = dataset["test"]
    #test_dataset = dataset["test"]
    #test_dataset = balanced_dataset
    # https://huggingface.co/datasets/cyberec/Prompt-injection-dataset
    #test_dataset = load_dataset("cyberec/Prompt-injection-dataset")
    
    '''
    test_dataset = load_dataset("JailbreakV-28K/JailBreakV-28K", "JailBreakV_28K")["JailBreakV_28K"]
    
    test_dataset = test_dataset.rename_column("jailbreak_query", "text")
    test_dataset = test_dataset.add_column("label", [1] * len(test_dataset))
    '''

    # adversarial, label, type
    # 1 = jb, 0 = not
    #test_dataset = load_dataset("allenai/wildjailbreak", "eval", delimiter="\t", keep_default_na=False)
    #test_dataset = test_dataset.rename_column("adversarial", "text")
    
    tokenized_test = test_dataset.map(tokenize_test)
    
    eval_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_eval_batch_size = 8,
        do_eval=True,
        disable_tqdm=True,
        report_to="none",
        seed=SEED,
    )
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    
    trainer = Trainer(
        model=model,
        args=eval_args,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )

    print(MODEL_NAME)
    metrics = trainer.evaluate()
    trainer.save_metrics(OUTPUT_DIR, metrics)

In [ ]:
def evaluate_threshold_helper(y_true, y_scores, thresholds=[0.5,0.6, 0.7, 0.8, 0.9, 0.95, 0.99]):
    for threshold in thresholds:
        y_pred = (y_scores >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        fpr = fp / (fp + tn) if (fp+tn) else 0.0

        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        print(f"=== {threshold} ===")
        print(f"FPR: {fpr} | Prec: {prec} | Rec: {rec} | f1: {f1}")
        print("-" * 50)
        


def evaluate_threshold(MODEL_NAME, OUTPUT_DIR):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=2, # Single output logit jb vs benign
    problem_type="single_label_classification",
    use_safetensors=True, # needed for sota model on JuptyerHub
    force_download=True,
    )
    
    def tokenize_test(text):
        return tokenizer(
            text["text"], 
            truncation=True, 
            max_length=512, 
        )
    
    #test_dataset = dataset["test"]
    test_dataset = balanced_dataset
    # https://huggingface.co/datasets/cyberec/Prompt-injection-dataset
    #test_dataset = load_dataset("cyberec/Prompt-injection-dataset")
    
    '''
    test_dataset = load_dataset("JailbreakV-28K/JailBreakV-28K", "JailBreakV_28K")["JailBreakV_28K"]
    
    test_dataset = test_dataset.rename_column("jailbreak_query", "text")
    test_dataset = test_dataset.add_column("label", [1] * len(test_dataset))
    '''

    # adversarial, label, type
    # 1 = jb, 0 = not
    #test_dataset = load_dataset("allenai/wildjailbreak", "eval", delimiter="\t", keep_default_na=False)
    #test_dataset = test_dataset.rename_column("adversarial", "text")
    
    tokenized_test = test_dataset.map(tokenize_test)
    
    eval_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_eval_batch_size = 8,
        do_eval=True,
        disable_tqdm=True,
        report_to="none",
        seed=SEED,
    )
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    
    trainer = Trainer(
        model=model,
        args=eval_args,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )

    prediction = trainer.predict(tokenized_test)
    #print(prediction)
    logits = prediction.predictions
    #print(logits)
    y_true = prediction.label_ids

    probs = softmax(logits, axis=-1)
    y_scores = probs[:, 1]
    print(y_scores[y_scores < 0.5])
    evaluate_threshold_helper(y_true, y_scores)


In [ ]:
"""
model_list = [
    ("roberta_harm_aug_detector/checkpoint-12297", "rob_harm_aug_balanced"),
    ("roberta_jailbreak", "rob_jb_balanced"),
    ("jackhhao/jailbreak-classifier", "jackhhao_jailbreak-classifier_balanced"),
    ("pmking27/jailbreak-detection", "pmking27_jailbreak-detection_balanced"),
    ("llm-semantic-router/mmbert32k-jailbreak-detector-merged", "llm-semenatic-router_mmbert32k_balanced") # CL
]

for t in model_list:
    model, output = t
    evaluate_model(model, output)

#shutil_helper("roberta_harm_aug_detector")
#ds = load_from_disk("combined_harm_datasets")
#ds_sanitycheck(ds)
"""

In [ ]:
# Example traiinings pipelin in jupterhub
"""
ds = combine_jb_datasets()
ds_sanitycheck(ds)
ds = load_from_disk("combined_input_aug")
ds = augment_ds_2(ds, "Old_scripts")
ds = resplit("combined_input_aug", "combined_input_aug_resplit")
ds_sanitycheck(ds)

train_model(
    MODEL_NAME="microsoft/deberta-v3-base",
    DATA_SET="combined_input_aug_resplit",
    OUTPUT_PATH="deberta_input_aug_resplit",
    EPOCHS=2)
    """

In [ ]:
##train_model(MODEL_NAME="FacebookAI/roberta-base", DATA_SET="combined_harm_aug", OUTPUT_PATH="./roberta_harm_aug_detector", EPOCHS=3)